<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/Physics%20Engine/2D/SIM%20V3/Indoor%20V3%20Sim/SIM%20V3_Indoor_Phase_B_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIM V3 · Indoor · Phase B — FDTD dataset

Generate the full-wave (FDTD) training boxes for the U-Net surrogate.
Ground truth = complex field `U(x)` from `FullWaveScene`, phase-reduced to
`Ũ = U·e^{+jkd}`. Indoor: the whole 7th-floor plane per Tx (cheap fields).

Output: `fw_data_indoor/shard_*.npz` (`x[9,H,W]`, `y[2,H,W]`).

In [10]:
# --- locate SIM V3 (works locally and on Colab) ---
# Colab: clone the repo, then set REPO_ROOT to it, e.g. '/content/Indoor_Walk_Test_7-7'.
import os, getpass
REPO_ROOT = '/content/indoor-walk-test'
if not os.path.isdir(REPO_ROOT):
    tok = getpass.getpass('GitHub token (repo read): ')      # not echoed
    os.system(f'git clone --depth 1 https://{tok}@github.com/cgm2179/indoor-walk-test.git "{REPO_ROOT}"')
# sanity: the file that was missing
print('dataset_3d present:',
      os.path.exists(os.path.join(REPO_ROOT, 'Physics Engine', '3D Map Physics', 'SIM V1 3D', 'dataset_3d.py')))

dataset_3d present: True


In [11]:
# Colab only: install deps (skip locally). torch usually preinstalled on Colab GPU.
# !pip -q install numpy scipy matplotlib tqdm onnxruntime
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

torch 2.11.0+cu128 | cuda True


### Parameters — scale `n_tx` / `boxes_per_field` up for a real model.

In [12]:
BANDS = ['LTE_B71_617', 'LTE_B13_751', 'LTE_B2_1960', 'NR_n41_2506', 'NR_n77_3700', 'WiFi_2G4', 'WiFi_5G']
SCENE = 'indoor'
N_TX = 6
BOXES_PER_FIELD = 80
BOX = 128
N_PER_WAVELENGTH = 8
REGION_M = 40   # outdoor only
OUT = '/content/drive/MyDrive/fw_data_indoor'


### Generate (this runs FDTD — the expensive, one-time step)

In [13]:
import sys
import os
# Temporarily print the output of a file search to debug the path issue
# !find "{REPO_ROOT}" -name "fw_dataset.py"
sys.path.append(os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')) # Corrected path
import fw_dataset
fw_dataset.generate(BANDS, scene=SCENE, n_tx=N_TX,
                    boxes_per_field=BOXES_PER_FIELD, box=BOX,
                    n_per_wavelength=N_PER_WAVELENGTH, region_m=REGION_M,
                    out_dir=OUT, seed=1)

field 617MHz: 100%|██████████| 9150/9150 [03:17<00:00, 46.31step/s]


  shard 000 indoor LTE_B71_617 tx=(131,64) boxes=80 x(9, 128, 128)


field 617MHz: 100%|██████████| 9150/9150 [03:17<00:00, 46.21step/s]


  shard 001 indoor LTE_B71_617 tx=(177,50) boxes=80 x(9, 128, 128)


field 617MHz: 100%|██████████| 9150/9150 [03:14<00:00, 47.02step/s]


  shard 002 indoor LTE_B71_617 tx=(235,71) boxes=80 x(9, 128, 128)


field 617MHz: 100%|██████████| 9150/9150 [03:17<00:00, 46.23step/s]


  shard 003 indoor LTE_B71_617 tx=(83,75) boxes=80 x(9, 128, 128)


field 617MHz:  37%|███▋      | 3355/9150 [01:07<01:57, 49.42step/s]


KeyboardInterrupt: 

### Inspect one training box (materials · |Ũ| target · log-distance)

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
d = np.load(sorted(glob.glob(OUT + '/shard_*.npz'))[0]); X, Y = d['x'], d['y']
i = 0; fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(np.argmax(X[i, :6], 0).T, origin='lower'); ax[0].set_title('materials')
ax[1].imshow(np.hypot(Y[i, 0], Y[i, 1]).T, origin='lower'); ax[1].set_title('|U~| (target)')
ax[2].imshow(X[i, 8].T, origin='lower'); ax[2].set_title('log-distance ch'); plt.show()
print('tensors:', X.shape, Y.shape)